In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

iris = load_iris()
x = iris.data
y = iris.target

print(x.shape)
print(iris.target_names)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print(x_train.shape)
print(x_test.shape)


def calculate_mean_std(x, y):
    mean = {}
    std = {}

    classes = np.unique(y)

    for c in classes:
        x_class = x[y == c]

        mean[c] = np.mean(x_class, axis=0)
        std[c] = np.std(x_class, axis=0)

    return mean, std


mean, std = calculate_mean_std(x_train, y_train)

def calculate_prior(y):
    classes, counts = np.unique(y, return_counts=True)

    prior = {}
    total = len(y)

    for c, count in zip(classes, counts):
        prior[c] = count / total

    return prior


prior = calculate_prior(y_train)

def gaussian_probability(x, mean, std):
    exponent = np.exp(-((x - mean) ** 2) / (2 * std ** 2))

    probability = (1 / (np.sqrt(2 * np.pi) * std)) * exponent

    return probability


def predict_sample(x):
    classes = np.unique(y_train)
    probabilities = {}

    for c in classes:

        prob = prior[c]

        for i in range(len(x)):
            prob *= gaussian_probability(
                x[i],
                mean[c][i],
                std[c][i]
            )

        probabilities[c] = prob

    return max(probabilities, key=probabilities.get)

y_pred = []

for sample in x_test:
    prediction = predict_sample(sample)
    y_pred.append(prediction)

y_pred = np.array(y_pred)

accuracy = accuracy_score(y_test, y_pred)

print("Predicted values:", y_pred)
print("Actual values:", y_test)
print("Accuracy:", accuracy)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

(150, 4)
['setosa' 'versicolor' 'virginica']
(120, 4)
(30, 4)
Predicted values: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Actual values: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Accuracy: 1.0
Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
